# One Piece Chapter 1177 — reMarkable 2 Landscape Spread

This notebook downloads One Piece Ch.1177 from WeebCentral and creates a **landscape double-page spread PDF** optimized for reMarkable 2.

- First page is kept **solo** (right side) so Oda's double spreads align correctly
- Pages are paired in manga reading order (right-to-left)
- Resolution: 1872×1404 @ 226 DPI (native reMarkable 2)

**Just run all cells!** No input needed.

In [ ]:
# Cell 1: Install dependencies & clone repo
!pip install -q requests 'httpx[http2]' nest_asyncio beautifulsoup4 lxml Pillow fpdf2 tqdm
!rm -rf /content/weebcentral_downloader
!git clone https://github.com/Yui007/weebcentral_downloader
print('\n✅ Setup complete!')

In [ ]:
# Cell 2: Download Chapter 1177
import sys
sys.path.insert(0, '/content/weebcentral_downloader/colab')

from colab_scraper import scrape_manga_info, scrape_chapter_list
from colab_downloader import download_chapters, parse_chapter_selection

SERIES_URL = 'https://weebcentral.com/series/01J76XY7E9FNDZ1DBBM6PBJPFK/One-Piece'
CHAPTER_NUM = 1177

manga_info = scrape_manga_info(SERIES_URL)
chapters = scrape_chapter_list(SERIES_URL)

# Find chapter 1177 by index
selected = [CHAPTER_NUM - 1]  # 0-based index
print(f'\n📖 Selected: {chapters[selected[0]]["title"]}')

# Download as images (we'll make our own PDF)
output_dir = download_chapters(
    manga_info=manga_info,
    chapters=chapters,
    selected_indices=selected,
    output_format='images',
    output_dir='/content/manga',
)
print(f'\n✅ Chapter downloaded to: {output_dir}')

In [ ]:
# Cell 3: Create landscape double-page spread PDF for reMarkable 2
import os
import glob
from PIL import Image

# reMarkable 2 native resolution in landscape
LANDSCAPE_WIDTH = 1872
LANDSCAPE_HEIGHT = 1404


def scale_to_fit(img, max_width, max_height):
    """Scale image to fit within max dimensions, preserving aspect ratio."""
    ratio = min(max_width / img.width, max_height / img.height)
    if ratio >= 1:
        return img
    new_size = (int(img.width * ratio), int(img.height * ratio))
    return img.resize(new_size, Image.LANCZOS)


def create_landscape_spread(image_dir, output_path):
    """
    Create landscape PDF with 2 pages per sheet.
    - Page 1 solo on right side
    - Then pairs: [2,3], [4,5], [6,7]...
    - Manga reading order: right page first, left page second
    """
    image_files = sorted(glob.glob(os.path.join(image_dir, '*')))
    image_files = [f for f in image_files
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.gif'))]

    if not image_files:
        print('ERROR: No images found!')
        return

    print(f'Found {len(image_files)} pages. Creating landscape spread...')

    # Build pairs: page 1 solo, then [2,3], [4,5], ...
    pairs = [(image_files[0], None)]  # First page solo
    i = 1
    while i < len(image_files):
        right = image_files[i]
        left = image_files[i + 1] if i + 1 < len(image_files) else None
        pairs.append((right, left))
        i += 2

    half_w = LANDSCAPE_WIDTH // 2
    sheets = []

    for right_path, left_path in pairs:
        sheet = Image.new('RGB', (LANDSCAPE_WIDTH, LANDSCAPE_HEIGHT), (255, 255, 255))

        # Right half (read first in manga)
        right_img = Image.open(right_path).convert('RGB')
        right_img = scale_to_fit(right_img, half_w, LANDSCAPE_HEIGHT)
        x = half_w + (half_w - right_img.width) // 2
        y = (LANDSCAPE_HEIGHT - right_img.height) // 2
        sheet.paste(right_img, (x, y))
        right_img.close()

        # Left half (read second in manga)
        if left_path:
            left_img = Image.open(left_path).convert('RGB')
            left_img = scale_to_fit(left_img, half_w, LANDSCAPE_HEIGHT)
            x = (half_w - left_img.width) // 2
            y = (LANDSCAPE_HEIGHT - left_img.height) // 2
            sheet.paste(left_img, (x, y))
            left_img.close()

        sheets.append(sheet)

    # Save as PDF
    sheets[0].save(
        output_path,
        'PDF',
        resolution=226.0,
        save_all=True,
        append_images=sheets[1:],
    )
    for s in sheets:
        s.close()

    print(f'\n✅ Landscape spread PDF created: {output_path}')
    print(f'   {len(sheets)} sheets total')
    print(f'   Page 1 solo on right side')
    print(f'   Remaining pages paired for double-spread viewing')


# Find the chapter image directory
manga_dir = output_dir
image_dirs = []
for entry in sorted(os.listdir(manga_dir)):
    entry_path = os.path.join(manga_dir, entry)
    if os.path.isdir(entry_path):
        images = [f for f in os.listdir(entry_path)
                  if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
        if images:
            image_dirs.append(entry_path)

if not image_dirs:
    print('ERROR: No image directories found!')
else:
    chapter_image_dir = image_dirs[-1]  # Latest/only chapter
    print(f'Using images from: {chapter_image_dir}')

    output_pdf = os.path.join(manga_dir, 'One_Piece_Ch1177_Landscape_Spread.pdf')
    create_landscape_spread(chapter_image_dir, output_pdf)

In [ ]:
# Cell 4: Download the PDF to your device
from google.colab import files

pdf_path = os.path.join(output_dir, 'One_Piece_Ch1177_Landscape_Spread.pdf')
if os.path.exists(pdf_path):
    print(f'📥 Downloading: {os.path.basename(pdf_path)}')
    print(f'   Size: {os.path.getsize(pdf_path) / 1024 / 1024:.1f} MB')
    files.download(pdf_path)
else:
    print('❌ PDF not found. Make sure Cell 3 ran successfully.')